In [1]:
# ============================================
# 1. IMPORTS AND SETUP
# ============================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from pathlib import Path

In [2]:
# ============================================
# 2. CONFIGURATION
# ============================================

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

RESULTS_DIR = Path("../results")

In [3]:
# ============================================
# 3. LOAD MERGED EMBEDDINGS
# ============================================

text = np.load(RESULTS_DIR / "text_embeddings_aligned.npy")
audio = np.load(RESULTS_DIR / "audio_embeddings_aligned.npy")
video = np.load(RESULTS_DIR / "video_embeddings_aligned.npy")
ids = np.load(RESULTS_DIR / "id_order.npy", allow_pickle=True)

N = text.shape[0]
print("N videos:", N, "device:", DEVICE)

N videos: 5136 device: cpu


In [4]:
# ============================================
# 4. TRAIN/TEST SPLIT
# ============================================

idx_all = np.arange(N)
idx_train, idx_test = train_test_split(idx_all, test_size=0.2, random_state=SEED)

In [5]:
# ============================================
# 5. STANDARDIZE EACH MODALITY
#    (Using Train Stats Only - No Data Leakage)
# ============================================

def fit_standardizer(mat, idx):
    mu = mat[idx].mean(axis=0, keepdims=True)
    sd = mat[idx].std(axis=0, keepdims=True) + 1e-6
    return mu, sd

def apply_standardizer(mat, mu, sd):
    return (mat - mu) / sd

t_mu, t_sd = fit_standardizer(text, idx_train)
a_mu, a_sd = fit_standardizer(audio, idx_train)
v_mu, v_sd = fit_standardizer(video, idx_train)

text_s = apply_standardizer(text, t_mu, t_sd).astype(np.float32)
audio_s = apply_standardizer(audio, a_mu, a_sd).astype(np.float32)
video_s = apply_standardizer(video, v_mu, v_sd).astype(np.float32)

text_t = torch.tensor(text_s, device=DEVICE)
audio_t = torch.tensor(audio_s, device=DEVICE)
video_t = torch.tensor(video_s, device=DEVICE)
idx_train_t = torch.tensor(idx_train, device=DEVICE)

In [6]:
# ============================================
# 6. MODEL DEFINITION
#    Projection Heads for Each Modality
# ============================================

SHARED_DIM = 128

class ProjectionHead(nn.Module):
    """Maps one modality's raw embedding into the shared 128-d space."""
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)  # L2-normalize -> dot product = cosine sim

text_head = ProjectionHead(text.shape[1], 256, SHARED_DIM).to(DEVICE)
audio_head = ProjectionHead(audio.shape[1], 512, SHARED_DIM).to(DEVICE)
video_head = ProjectionHead(video.shape[1], 512, SHARED_DIM).to(DEVICE)

In [7]:
# ============================================
# 7. TRAINING SETUP
#    Optimizer, Loss Function, Batch Iterator
# ============================================

params = list(text_head.parameters()) + list(audio_head.parameters()) + list(video_head.parameters())
opt = torch.optim.Adam(params, lr=1e-3, weight_decay=1e-5)
TEMPERATURE = 0.07

def info_nce(z1, z2, temperature):
    """Symmetric contrastive loss: pulls same-video pairs together, pushes others apart."""
    logits = z1 @ z2.T / temperature
    labels = torch.arange(z1.shape[0], device=z1.device)
    loss_a = F.cross_entropy(logits, labels)
    loss_b = F.cross_entropy(logits.T, labels)
    return (loss_a + loss_b) / 2

def batch_iter(idx_tensor, batch_size):
    perm = idx_tensor[torch.randperm(len(idx_tensor))]
    for i in range(0, len(perm), batch_size):
        yield perm[i:i + batch_size]

EPOCHS = 60
BATCH_SIZE = 256

In [8]:
# ============================================
# 8. TRAINING LOOP
# ============================================

for epoch in range(1, EPOCHS + 1):
    text_head.train(); audio_head.train(); video_head.train()
    total_loss, n_batches = 0.0, 0
    for batch_idx in batch_iter(idx_train_t, BATCH_SIZE):
        if len(batch_idx) < 2:
            continue
        zt = text_head(text_t[batch_idx])
        za = audio_head(audio_t[batch_idx])
        zv = video_head(video_t[batch_idx])

        loss = (
            info_nce(zt, za, TEMPERATURE) +
            info_nce(zt, zv, TEMPERATURE) +
            info_nce(za, zv, TEMPERATURE)
        ) / 3

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item()
        n_batches += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f"epoch {epoch:3d}  train_loss={total_loss / max(n_batches,1):.4f}")

epoch   1  train_loss=5.1404
epoch  10  train_loss=0.6460
epoch  20  train_loss=0.1959
epoch  30  train_loss=0.1158
epoch  40  train_loss=0.0872
epoch  50  train_loss=0.0726
epoch  60  train_loss=0.0649


In [9]:
# ============================================
# 9. SAVE MODEL
# ============================================

torch.save({
    "text_head": text_head.state_dict(),
    "audio_head": audio_head.state_dict(),
    "video_head": video_head.state_dict(),
    "t_mu": t_mu, "t_sd": t_sd,
    "a_mu": a_mu, "a_sd": a_sd,
    "v_mu": v_mu, "v_sd": v_sd,
    "idx_train": idx_train, "idx_test": idx_test,
}, RESULTS_DIR / "alignment_model.pt")

print("Saved alignment_model.pt")

Saved alignment_model.pt
